# Robust Nonconvex Optimization

This notebook solves the direct nonlinear robust formulation from the original `robust_nonconvex.ipynb`. The solver details are now in `mechanism_design/nonconvex.py`.

`CasADi` with IPOPT is the default solver. Change OT_RADIUS and fhat to see the results for different wasserstein balls.


In [1]:
from pathlib import Path

import numpy as np

from mechanism_design.data import DATASET_DIR_NAME, build_load_scenario
from mechanism_design.nonconvex import (
    NonconvexLayout,
    build_type_distance_matrix,
    check_nonconvex_initialization,
    initial_nonconvex_point,
    solve_nonconvex_robust,
)


## Experiment Inputs

These defaults match the original nonconvex notebook: price scale `2.`, `Dreq` sampled between `100` and `200`.

In [2]:
M = 80
T = 24
N = 5

#OT_RADIUS = 0.1
OT_RADIUS = 0.8
NONCONVEX_SOLVER = "casadi"

scenario = build_load_scenario(
    M=M,
    T=T,
    N=N,
    folder=Path(DATASET_DIR_NAME),
    price_scale=2.0,
    dreq_low=100.0,
    dreq_high=200.0,
    seed=42,
)

fhat = np.array([0.10, 0.15, 0.20, 0.25, 0.30], dtype=float)
#fhat = np.array([0.30, 0.25, 0.20, 0.15, 0.10], dtype=float)
C = build_type_distance_matrix(N)

print(f"Loaded {len(scenario.files)} CSV files from: {scenario.folder}")
print("d_n shape:", scenario.d_n.shape)
print("Feasibility report:", scenario.feasibility_report)
print("Dreq:", np.round(scenario.Dreq, 4))
print("Upper bound M*N*Krt:", np.round(scenario.M * scenario.N * scenario.Krt, 4))


Loaded 5 CSV files from: Dataset on Hourly Load Profiles for 8 Facilities (8760 hours)
d_n shape: (5, 24)
Feasibility report: {'sum_Krt': 2528.4, 'Ks': 2571.13, 'gap': 42.73000000000002, 'daily_cap_covers_hourly_caps': True}
Dreq: [177.3956 143.8878 185.8598 169.7368 109.4177 197.5622 176.114  178.6064
 112.8114 145.0386 137.0798 192.6765 164.3865 182.2762 144.3414 122.7239
 155.4585 106.3817 182.7631 163.1664 175.8088 135.4526 197.0698 189.3121]
Upper bound M*N*Krt: [20000. 24800. 20000. 24800. 20000. 24800. 20000. 24800. 61648. 70624.
 86400. 91056. 65296. 70032. 85952. 90752. 31600. 27600. 22800. 27600.
 22800. 27600. 22800. 27600.]


In [3]:
fhat

array([0.1 , 0.15, 0.2 , 0.25, 0.3 ])

## Initial Point Check

The starting point keeps the transport plans diagonal. This is feasible for `r=0` and remains feasible for positive radii.


In [4]:
layout = NonconvexLayout(T=scenario.T, N=scenario.N)
initial = initial_nonconvex_point(layout, fhat=fhat, x_value=0.1)
v0 = layout.pack(*initial, order="C")

print("Total decision variables:", layout.n_vars)
print(check_nonconvex_initialization(layout=layout, fhat=fhat, C=C, r=OT_RADIUS, vector=v0))


Total decision variables: 1014
{'z_column_residual': 0.0, 'y_column_residual': 0.0, 'z_transport_cost': 0.0, 'max_y_transport_slack': -0.08000000000000002}


## Solve

`solve_nonconvex_robust` accepts `solver="casadi"`. CasADi is now the default because IPOPT is generally better suited for this smooth constrained nonlinear program.


In [5]:

result = solve_nonconvex_robust(
    M=scenario.M,
    T=scenario.T,
    N=scenario.N,
    Krt=scenario.Krt,
    Ks=scenario.Ks,
    Dreq=scenario.Dreq,
    pi=scenario.pi,
    alpha=scenario.alpha,
    beta=scenario.beta,
    r=OT_RADIUS,
    fhat=fhat,
    C=C,
    solver=NONCONVEX_SOLVER,
    verbose=True,
)

D_star = result.D

print("success:", result.success)
print("solver:", result.solver)
print("status:", result.status)
print("objective:", result.objective)
print("Zi*:", np.round(result.Zi, 6))
print("x* first 6:", np.round(result.x[:6], 6))
print("D* shape:", D_star.shape, "(T,N)")
print("D* first hour:", np.round(D_star[0], 6))



******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:      745
Number of nonzeros in inequality constraint Jacobian.:      524
Number of nonzeros in Lagrangian Hessian.............:    21453

Total number of variables............................:     1014
                     variables with only lower bounds:     1014
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:      125
Total number of inequality c

## Inspect the Recovered Reduction Matrix

Rows are hours and columns are types, matching the original notebook's recovered `D_star` shape.


In [6]:
D_star


array([[3.24999952, 2.89497243, 2.57663385, 2.31626018, 2.10461103],
       [3.18529382, 2.83683071, 2.52437724, 2.26881696, 2.06108076],
       [3.15257642, 2.90125486, 2.64994706, 2.39863971, 2.14733247],
       [3.07322358, 2.7361277 , 2.43386819, 2.18664927, 1.98568664],
       [3.0334923 , 2.70042734, 2.40178082, 2.15751344, 1.95895686],
       [3.34598824, 3.08039359, 2.8147998 , 2.5492061 , 2.28361243],
       [3.00004499, 2.7516824 , 2.51243954, 2.2731974 , 2.03395542],
       [3.03269092, 2.79022201, 2.5477689 , 2.30531628, 2.06286378],
       [3.03349231, 2.70042734, 2.40178082, 2.15751344, 1.95895686],
       [3.07322202, 2.73612698, 2.43386688, 2.18664392, 1.98568484],
       [3.12499862, 2.78265161, 2.47568222, 2.22460746, 2.02051738],
       [3.26523747, 3.00560373, 2.74597426, 2.48634508, 2.22671598],
       [3.24999871, 2.89497209, 2.57663346, 2.31625963, 2.10461067],
       [3.31470443, 2.95311387, 2.62889025, 2.36370334, 2.14814111],
       [3.37499843, 3.00729246, 2.